- fix network size
- fix dim d

take one of
- grad u
- laplace u
- div(A\nabla u)

compute the mem required and time spend for:
- forward pass:
```python
u = model(X)
```
- deriv comp:
```python
grad_u = compute_grad()
```


do this a few times and calc average + stand dev


do this for many network sizes and dims d

In [1]:
import torch
import derivatives, architecture
import utility

In [2]:
d = 4
D = d+1
layers = [64,64]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
model = architecture.PINN(D, layers, 1)
model
#model = torch.compile(model)

PINN(
  (act): Tanh()
  (net): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [17]:
n_steps = 10
n_res = 10_000

der_mode = 'lapl'
der_mode = 'div'
der_mode = 'grad'

from torch.profiler import profile, ProfilerActivity
from torch.profiler import record_function
schedule = torch.profiler.schedule(wait=2, warmup=2, active=3, repeat=1)
prof_ctx = profile(
    activities=[ProfilerActivity.CPU],
    profile_memory=True,
    record_shapes=True,
    with_stack=True,
    schedule=schedule
)

n_steps_warm_up = 0
for s in range(n_steps+n_steps_warm_up):

    if s == n_steps_warm_up:
        prof_ctx.__enter__()

    with record_function("sample"):
        X = torch.rand(n_res, D, dtype=torch.float32, device=device)
    X.requires_grad = True
    #print(X.shape)

    with record_function("forward-pass"):
        u = model(X)
    #print(u.shape)

    with record_function("der-comp"):
        if der_mode == 'grad':
            u_grad = derivatives.compute_grad(X, u, torch.ones_like(u))
        elif der_mode == 'lapl':
            u, grad_u, spatial_laplace_u = derivatives.compute_derivatives(model, X)
        elif der_mode == 'div':
            u_grad = derivatives.compute_grad(X, u, torch.ones_like(u))
            u_t = u_grad[:,-1:]
            u_jac = u_grad[:,:-1]
            model_u_grad = lambda X_in: u_jac
            s, jac = derivatives.compute_score_and_jacobian(model_u_grad, X)

    if s >= n_steps_warm_up:
        prof_ctx.step()

prof_ctx.__exit__(None, None, None)
# save results
prof_ctx.export_chrome_trace("prof_trace.json")
report = prof_ctx.key_averages().table(sort_by="cpu_time_total", row_limit=20)
print(report)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         2.12%     274.283us       100.00%      12.968ms       4.323ms      15.11 MB           0 B             3  
                                               der-comp        10.47%       1.357ms        52.78%       6.845ms       2.282ms    -156.25 KB     -30.14 MB             3  
                                           forward-pass         1.60%     206.970us        39.75%       5.155ms       1.718ms      14.69 MB     -78.12

profiling guide:
https://huggingface.co/blog/torch-profiler

- The "Self" columns measure time spent only inside the event itself, excluding its children.
- The "total" columns include the event and all of its children together.

analyze the .json file with:
https://ui.perfetto.dev/

In [11]:
prof_ctx.key_averages()

[<FunctionEventAvg key=ProfilerStep* self_cpu_time=290.088us cpu_time=4.724ms  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=13280000 cuda_memory_usage=0>,
 <FunctionEventAvg key=sample self_cpu_time=213.842us cpu_time=243.057us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=600000 cuda_memory_usage=0>,
 <FunctionEventAvg key=aten::rand self_cpu_time=37.759us cpu_time=171.776us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=600000 cuda_memory_usage=0>,
 <FunctionEventAvg key=aten::empty self_cpu_time=26.660us cpu_time=8.887us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=600000 cuda_memory_usage=0>,
 <FunctionEventAvg key=aten::uniform_ self_cpu_time=450.910us cpu_time=150.303us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=0 cuda_memory_usage=0>,
 <FunctionEventAvg key=forward-pass self_cpu_time=277.521us cpu_time=2.115ms  self_cuda_time=0.000us cuda_time

In [18]:
my_events = []
prof_ctx.key_averages()[0].key
for event in prof_ctx.key_averages():
    if event.key in ["sample", "forward-pass", "der-comp"]:
        my_events.append(event)
my_events

[<FunctionEventAvg key=sample self_cpu_time=196.260us cpu_time=231.546us  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=600000 cuda_memory_usage=0>,
 <FunctionEventAvg key=forward-pass self_cpu_time=206.970us cpu_time=1.718ms  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=15400000 cuda_memory_usage=0>,
 <FunctionEventAvg key=der-comp self_cpu_time=1.357ms cpu_time=2.282ms  self_cuda_time=0.000us cuda_time=0.000us input_shapes= cpu_memory_usage=-160000 cuda_memory_usage=0>]

In [20]:
for ev in my_events:
    print(f"{ev.key}")
    print(f"""\
- cpu_time={ev.cpu_time:f}
- cpu_mem_usage={ev.cpu_memory_usage/1024}
- count={ev.count}
""")

sample
- cpu_time=231.546000
- cpu_mem_usage=585.9375
- count=3

forward-pass
- cpu_time=1718.311000
- cpu_mem_usage=15039.0625
- count=3

der-comp
- cpu_time=2281.501000
- cpu_mem_usage=-156.25
- count=3



In [14]:
print(report)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         2.05%     290.088us       100.00%      14.172ms       4.724ms      12.66 MB           0 B             3  
                                               der-comp        11.05%       1.566ms        48.05%       6.809ms       2.270ms      -2.59 MB     -32.58 MB             3  
                                           forward-pass         1.96%     277.521us        44.76%       6.344ms       2.115ms      14.69 MB     -78.12